# Reusable Template: Regularized Linear Model Pipeline

A parameterized notebook you can drop into a new project. It supports **both regression** (Ridge/Lasso/ElasticNet on a continuous target) **and classification** (regularized LogisticRegression on a binary target) — just flip the `TASK` setting in the config cell.

**How to reuse this for a new project:**
1. Point `CSV_PATH` at your file and set `TARGET_COL`.
2. Set `TASK` to `'classification'` or `'regression'`.
3. Run all cells top to bottom.
4. Adjust the hyperparameter search ranges in the config cell if your first pass suggests the optimum is at the edge of the grid.

This template is demonstrated below on `wine_quality.csv`, but nothing except the config cell is dataset-specific.

## 1. Configuration — edit this cell for your project

In [ ]:
CONFIG = {
    'CSV_PATH': 'wine_quality.csv',
    'TARGET_COL': 'quality',
    'TASK': 'classification',        # 'classification' or 'regression'
    'TEST_SIZE': 0.2,
    'RANDOM_STATE': 42,
    'STRATIFY': True,                # only used for classification
    'C_OR_ALPHA_RANGE': (-4, 2),     # exponents for np.logspace; C for classification, alpha for regression
    'N_GRID_POINTS': 100,
    'CV_FOLDS': 5,
    'CLASSIFICATION_SCORING': 'f1',
    'REGRESSION_SCORING': 'neg_mean_squared_error',
}
CONFIG

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge, Lasso, LogisticRegression
from sklearn.metrics import (
    mean_squared_error, r2_score,
    f1_score, accuracy_score, precision_score, recall_score, roc_auc_score,
    ConfusionMatrixDisplay,
)

## 3. Load & prepare data

In [ ]:
def load_data(config):
    df = pd.read_csv(config['CSV_PATH'])
    y = df[config['TARGET_COL']]
    X_raw = df.drop(columns=[config['TARGET_COL']])
    return df, X_raw, y

def scale_and_split(X_raw, y, config):
    scaler = StandardScaler().fit(X_raw)
    X = scaler.transform(X_raw)
    strat = y if (config['TASK'] == 'classification' and config['STRATIFY']) else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=config['TEST_SIZE'],
        random_state=config['RANDOM_STATE'], stratify=strat
    )
    return X_train, X_test, y_train, y_test, scaler

df, X_raw, y = load_data(CONFIG)
X_train, X_test, y_train, y_test, scaler = scale_and_split(X_raw, y, CONFIG)
predictors = X_raw.columns
print(df.shape, '-> train:', X_train.shape, 'test:', X_test.shape)

## 4. Baseline model (no regularization) — sanity check for overfitting

In [ ]:
def fit_baseline(X_train, y_train, config):
    if config['TASK'] == 'classification':
        model = LogisticRegression(penalty=None, max_iter=5000)
    else:
        from sklearn.linear_model import LinearRegression
        model = LinearRegression()
    model.fit(X_train, y_train)
    return model

def evaluate(model, X, y, task, label=''):
    pred = model.predict(X)
    if task == 'classification':
        return {'label': label, 'accuracy': accuracy_score(y, pred), 'f1': f1_score(y, pred)}
    else:
        return {'label': label, 'mse': mean_squared_error(y, pred), 'r2': r2_score(y, pred)}

baseline = fit_baseline(X_train, y_train, CONFIG)
print(evaluate(baseline, X_train, y_train, CONFIG['TASK'], 'train'))
print(evaluate(baseline, X_test, y_test, CONFIG['TASK'], 'test'))

## 5. Regularized model + hyperparameter tuning

In [ ]:
def build_regularized_estimator(config, penalty='l2'):
    if config['TASK'] == 'classification':
        kwargs = dict(max_iter=5000)
        if penalty == 'l1':
            kwargs.update(penalty='l1', solver='liblinear')
        elif penalty == 'elasticnet':
            kwargs.update(penalty='elasticnet', solver='saga', l1_ratio=0.5)
        else:
            kwargs.update(penalty='l2')
        return LogisticRegression(**kwargs)
    else:
        return Lasso(max_iter=5000) if penalty == 'l1' else Ridge()

def tune(config, penalty='l2'):
    estimator = build_regularized_estimator(config, penalty)
    param_name = 'C' if config['TASK'] == 'classification' else 'alpha'
    param_grid = {param_name: np.logspace(*config['C_OR_ALPHA_RANGE'], config['N_GRID_POINTS'])}
    scoring = config['CLASSIFICATION_SCORING'] if config['TASK'] == 'classification' else config['REGRESSION_SCORING']
    gs = GridSearchCV(estimator, param_grid=param_grid, scoring=scoring, cv=config['CV_FOLDS'])
    gs.fit(X_train, y_train)
    return gs

gs_l2 = tune(CONFIG, penalty='l2')
print('Best params:', gs_l2.best_params_, '| best CV score:', gs_l2.best_score_)

## 6. Refit best model & evaluate on held-out test set

In [ ]:
best_model = gs_l2.best_estimator_
print('--- Final evaluation ---')
print(evaluate(best_model, X_train, y_train, CONFIG['TASK'], 'train'))
print(evaluate(best_model, X_test, y_test, CONFIG['TASK'], 'test'))

if CONFIG['TASK'] == 'classification':
    ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test)
    plt.show()

## 7. Coefficient plot

In [ ]:
coef = pd.Series(np.ravel(best_model.coef_), predictors).sort_values()
coef.plot(kind='bar', title='Tuned model coefficients')
plt.axhline(0, color='k', linewidth=0.8)
plt.tight_layout()
plt.show()

## 8. Compare L1 vs. L2 vs. Elastic Net (classification) or Ridge vs. Lasso (regression)

In [ ]:
penalties = ['l2', 'l1'] if CONFIG['TASK'] == 'classification' else ['l2', 'l1']
results = []
for pen in penalties:
    g = tune(CONFIG, penalty=pen)
    m = g.best_estimator_
    row = evaluate(m, X_test, y_test, CONFIG['TASK'], pen)
    row['best_hyperparam'] = list(g.best_params_.values())[0]
    results.append(row)

pd.DataFrame(results)

## 9. Adapting this template

- **New dataset:** change `CSV_PATH` / `TARGET_COL` in the config cell only.
- **More features than samples, or many irrelevant features:** default to `penalty='l1'` in step 5/8 and inspect the coefficient plot for zeros.
- **Multi-class target:** `LogisticRegression` handles this automatically (`multi_class='auto'`); the coefficient plot in step 7 will need reshaping — `model.coef_` becomes `(n_classes, n_features)`.
- **Categorical features:** one-hot encode them before scaling (`pd.get_dummies` or `ColumnTransformer` + `OneHotEncoder`) — `StandardScaler` assumes numeric input.
- **Bigger data / need speed:** switch `solver` to `'saga'` and consider `SGDClassifier(penalty=...)` for very large datasets.
- **Want nested CV to avoid optimistic hyperparameter estimates:** wrap the `tune()` call's `GridSearchCV` in an outer `cross_val_score` loop, or use `sklearn.model_selection.cross_validate` with `GridSearchCV` as the estimator.